# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [3]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import numpy as np
from IPython.display import display

# Find the CSV beside the notebook/current working directory.
candidates = [Path.cwd() / "world_population.csv", Path("/mnt/data/world_population.csv")]
csv_path = next((p for p in candidates if p.exists()), None)

# If running in Google Colab and the CSV is not found, upload it when prompted.
if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = Path(next(iter(uploaded))) if uploaded else None
    except ImportError:
        pass

if csv_path is None or not csv_path.exists():
    raise FileNotFoundError(
        "Cannot find world_population.csv. Put it in the same folder as this notebook "
        "and run this cell again."
    )

df = pd.read_csv(csv_path)

# Match the column names used in the quiz.
df = df.rename(columns={
    'Country/Territory': 'Country',
    'Area (km²)': 'Area',
    'Density (per km²)': 'Density',
    'World Population Percentage': 'World share (%)'
})

# Long/tidy version: one row per country per year.
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(
    id_vars=['CCA3', 'Country', 'Continent'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Population'
)
df_long['Year'] = df_long['Year'].str[:4].astype(int)

print(f"Loaded: {csv_path}")
print("Country data shape:", df.shape)
print("Long data shape:", df_long.shape)


Saving world_population.csv to world_population.csv
Loaded: world_population.csv
Country data shape: (234, 16)
Long data shape: (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

A line chart works best here because it shows how each continent changed over time with all of them starting from the point. I set each continent to 100 in 1970. The y-axis shows how much each one grew relative to that starting point. This way the chart focuses on growth rates of the actual population numbers, which are very different, between continents. The title points out the result: Africa grew the fastest.

In [4]:
# Q1 — Relative population growth by continent
year_cols = [c for c in df.columns if c.endswith('Population')]
continent_year = df.groupby('Continent')[year_cols].sum()
years = [int(c[:4]) for c in year_cols]
growth_index = continent_year.div(continent_year['1970 Population'], axis=0) * 100
growth_long = growth_index.reset_index().melt(id_vars='Continent', var_name='Year', value_name='Index')
growth_long['Year'] = growth_long['Year'].str[:4].astype(int)
growth_long = growth_long.sort_values('Year')

fig = px.line(growth_long, x='Year', y='Index', color='Continent',
              markers=True, title='Africa’s population grew fastest since 1970',
              labels={'Index': 'Population index (1970 = 100)', 'Year': 'Year'},
              color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(template='plotly_white', legend_title_text='', yaxis_title='Population index (1970 = 100)',
                  xaxis_title='', hovermode='x unified')
fig.update_xaxes(showgrid=False, dtick=10)
fig.update_yaxes(showgrid=True, gridcolor='#E8E8E8', rangemode='tozero')
fig.show()

continent_growth = (continent_year['2022 Population'] / continent_year['1970 Population'] - 1) * 100
display(continent_growth.sort_values(ascending=False).rename('Growth 1970–2022 (%)').round(1).to_frame())

,Growth 1970–2022 (%)
Continent,
Africa,290.4
Oceania,131.2
South America,126.4
Asia,120.1
North America,90.3
Europe,13.3


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

A sorted horizontal bar chart makes it easy to compare the ranking and the size of each countrys contribution. I use grey for countries and one accent colour for African countries. This way the continent membership is clear, without distracting from the ranking.

In [5]:
# Q2 — Top 15 countries by absolute population added, 1970–2022
top15 = df.assign(People_added=df['2022 Population'] - df['1970 Population']).nlargest(15, 'People_added').sort_values('People_added')
top15['Highlight'] = np.where(top15['Continent'].eq('Africa'), 'Africa', 'Other continents')

fig = px.bar(top15, x='People_added', y='Country', orientation='h', color='Highlight',
             color_discrete_map={'Africa': '#D97706', 'Other continents': '#9CA3AF'},
             title='India and China added the most people; four top-15 countries are African',
             labels={'People_added': 'People added, 1970–2022', 'Country': '', 'Highlight': ''},
             text=top15['People_added'].map(lambda v: f'{v/1e6:.0f}M'))
fig.update_layout(template='plotly_white', legend_title_text='', xaxis_title='People added',
                  yaxis_title='', showlegend=True)
fig.update_xaxes(showgrid=False, tickformat=',')
fig.update_yaxes(showgrid=False)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.show()

print(f"African countries in the top 15: {(top15['Continent'] == 'Africa').sum()}")

African countries in the top 15: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

A scatter plot shows how land area and population are related while keeping every country visible. Logarithmic axes are the choice because both land area and population cover many orders of magnitude. The scatter plot uses colour to show continent. The scatter plot gives labels that point out the unusually dense and sparsely populated cases.

In [8]:
import plotly.express as px

fig = px.scatter(
    df,
    x="Area",
    y="2022 Population",
    color="Continent",
    hover_name="Country",
    log_x=True,
    log_y=True,
    title="Land Area vs 2022 Population"
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

The box plot shows that population density varies considerably within and across continents. Asia and Europe have relatively high median densities, while South America has a lower median. Several countries appear as outliers, indicating unusually high population density compared with other countries in their continents. The logarithmic scale helps compare countries with very different density values.

In [13]:
fig = px.box(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    points='outliers',
    log_y=True,
    title='Population Density Varies Widely Within and Across Continents',
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='Continent',
    yaxis_title='Population Density (people per km², log scale)',
    showlegend=False,
    yaxis_showgrid=False
)

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

A choropleth map is the way to show where population growth and where population decline happen in different places. The blue and orange color scale shows which areas are getting smaller and which are getting bigger with the middle being zero. The percentage change helps compare countries that're different in size.

In [10]:
import plotly.express as px

# Calculate percentage population change
df['Change (%)'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population'] * 100
)

# Create world map
fig = px.choropleth(
    df,
    locations='CCA3',
    color='Change (%)',
    hover_name='Country',
    locationmode='ISO-3',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    title='Population Growth and Decline by Country (2000–2022)'
)

fig.update_layout(template='plotly_white')
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

A treemap helps show how parts relate to the whole in a way. The top level groups countries, by continent. Inside each continent smaller rectangles represent countries. The size of each rectangle shows how much that country contributes to the world population in 2022. Bigger rectangles mean populations. This layout makes it easy to compare both continents and countries at the time.

In [11]:
import plotly.express as px

fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    title="World Population by Continent and Country (2022)"
)

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

I see a 100% stacked bar chart that shows how the worlds population is made up each year. The 100% stacked bar chart uses eight years. Every bar adds up to 100 percent. This lets me compare how the share of Africa, Asia, Europe and the remaining parts have changed, without being affected by how the total world population grows.

In [12]:
import plotly.express as px

data = df_long.groupby(['Year', 'Continent'])['Population'].sum().reset_index()

data['Continent'] = data['Continent'].replace(
    ['Oceania', 'North America', 'South America'],
    'Rest of the world'
)

data = data.groupby(['Year', 'Continent'])['Population'].sum().reset_index()

fig = px.area(
    data,
    x='Year',
    y='Population',
    color='Continent',
    title='World Population Share by Continent'
)

fig.show()